# Phase 2 — Gradient Boosting Model: Hypertension Prediction (RIPAS Dataset)
**Triage IQ — Predicting Hypertension from Emergency Department Triage Data**

This notebook implements Gradient Boosting as a competing model to XGBoost for RIPAS hypertension prediction.  
The same pipeline, preprocessing, and evaluation protocol are used to ensure a fair comparison.

**Dataset:** `ripas_dataset.csv`  
**Target variable:** `HTN/CHD` (clinically recorded, binary)

## 1. Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os

from sklearn.model_selection import (
    train_test_split, StratifiedKFold,
    RandomizedSearchCV, GridSearchCV,
    RepeatedStratifiedKFold, cross_validate
)
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    accuracy_score, roc_auc_score, recall_score,
    classification_report, confusion_matrix,
    ConfusionMatrixDisplay, roc_curve
)

from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import SMOTE

from sklearn.ensemble import GradientBoostingClassifier

import warnings
warnings.filterwarnings("ignore")

## 2. Load Dataset

In [ ]:
df = pd.read_csv("ripas_dataset.csv")

print("Shape:", df.shape)
df.head()

## 3. Data Cleaning

In [ ]:
# Strip trailing 'Y' from age strings (e.g. '45Y' -> 45)
df['age'] = pd.to_numeric(
    df['age'].astype(str).str.replace('Y', '', regex=False),
    errors='coerce'
)

# Convert vital-sign and stay columns to numeric
numeric_cols = [
    'Blood pressure diastole',
    'blood pressure/systolic',
    'pulse rate',
    'respiratory rate',
    'SPO2',
    'Temperature',
    'PAIN SCORE',
    'STAYS IN DAYS'
]
for col in numeric_cols:
    df[col] = pd.to_numeric(df[col], errors='coerce')

# Encode binary comorbidity / outcome flags (NO -> 0, named value -> 1)
binary_cols = ['HTN/CHD', 'DM', 'CKD', 'ADMISSION', 'ICU', 'NIV/VENT', 'INOTROPE', 'DEATH']
for col in binary_cols:
    df[col] = df[col].replace({'NO': 0, 'HTN/CHD': 1, 'DM': 1, 'CKD': 1})
    df[col] = pd.to_numeric(df[col], errors='coerce')

# Encode gender
df['gender'] = df['gender'].map({'M': 1, 'F': 0})

print("Cleaned dataset shape:", df.shape)
df.head()

## 4. Define Target Variable

In [ ]:
# Drop rows where the target label is missing
df = df[df['HTN/CHD'].notna()].reset_index(drop=True)

print("Remaining NaN in target:", df['HTN/CHD'].isna().sum())

X = df.drop('HTN/CHD', axis=1)
y = df['HTN/CHD']

print("\nClass distribution:")
print(y.value_counts())
print("\nClass proportions:")
print(y.value_counts(normalize=True).round(3))

## 5. Train/Test Split

In [ ]:
# 80/20 stratified split — preserves class ratio in both sets
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print(f"Training set: {X_train.shape[0]} samples")
print(f"Test set:     {X_test.shape[0]} samples")
print("\nTraining class distribution:")
print(y_train.value_counts())

## 6. Class Distribution and SMOTE Effect

In [ ]:
# Visualise class imbalance before any resampling
fig, ax = plt.subplots(figsize=(5, 3))
y_train.value_counts().plot(kind='bar', ax=ax, color=['steelblue', 'coral'])
ax.set_xticklabels(['No HTN', 'HTN'], rotation=0)
ax.set_title("Training Set Class Distribution (Before SMOTE)")
ax.set_ylabel("Count")
plt.tight_layout()
plt.show()

In [ ]:
categorical_cols = ['site of pain']
numerical_cols   = [col for col in X.columns if col not in categorical_cols]

preprocessor = ColumnTransformer(
    transformers=[
        ('num', Pipeline([
            ('imputer', SimpleImputer(strategy='median')),
            ('scaler',  StandardScaler())
        ]), numerical_cols),

        ('cat', Pipeline([
            ('imputer', SimpleImputer(strategy='most_frequent')),
            ('encoder', OneHotEncoder(drop='first', handle_unknown='ignore'))
        ]), categorical_cols)
    ]
)

In [ ]:
# Illustrate SMOTE effect by fitting preprocessor + SMOTE on training data
X_pre = preprocessor.fit_transform(X_train)
_, y_res = SMOTE(sampling_strategy=0.8, random_state=42).fit_resample(X_pre, y_train)

print("Class distribution after SMOTE:")
print(pd.Series(y_res).value_counts())

fig, ax = plt.subplots(figsize=(5, 3))
pd.Series(y_res).value_counts().plot(kind='bar', ax=ax, color=['steelblue', 'coral'])
ax.set_xticklabels(['No HTN', 'HTN'], rotation=0)
ax.set_title("Training Set Class Distribution (After SMOTE, sampling_strategy=0.8)")
ax.set_ylabel("Count")
plt.tight_layout()
plt.show()

## 7. Gradient Boosting Pipeline

In [ ]:
gb = GradientBoostingClassifier(random_state=42)

pipeline = ImbPipeline(steps=[
    ('preprocessor', preprocessor),
    ('smote',        SMOTE(sampling_strategy=0.8, random_state=42)),
    ('model',        gb)
])

## 8. Baseline Evaluation

Repeated stratified CV establishes the performance floor before any tuning.

In [ ]:
rskf = RepeatedStratifiedKFold(n_splits=5, n_repeats=5, random_state=42)
scoring = {'recall': 'recall', 'roc_auc': 'roc_auc', 'precision': 'precision', 'f1': 'f1'}

cv_results = cross_validate(pipeline, X_train, y_train, cv=rskf, scoring=scoring)

print("Repeated Stratified CV — Baseline Gradient Boosting:")


In [ ]:
for key in cv_results:
    if key.startswith("test_"):
        mean_score = np.mean(cv_results[key])
        std_score  = np.std(cv_results[key])
        print(f"{key[5:]:12s}: {mean_score:.4f} ± {std_score:.4f}")

In [ ]:
pipeline.fit(X_train, y_train)
y_prob_base = pipeline.predict_proba(X_test)[:, 1]
y_pred_base = (y_prob_base >= 0.35).astype(int)

print("--- BASELINE (Threshold = 0.35) ---")
print(f"Accuracy : {accuracy_score(y_test, y_pred_base):.3f}")
print(f"ROC-AUC  : {roc_auc_score(y_test, y_prob_base):.3f}")
print(classification_report(y_test, y_pred_base))

## 9. Hyperparameter Tuning

### 9a. RandomizedSearchCV

In [ ]:
cv_rand = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)

param_dist = {
    'model__n_estimators':  [100, 200, 300],
    'model__max_depth':     [3, 5, 7],
    'model__learning_rate': [0.01, 0.1, 0.2],
    'model__subsample':     [0.8, 1.0]
}

search = RandomizedSearchCV(
    pipeline,
    param_distributions=param_dist,
    n_iter=20,
    cv=cv_rand,
    scoring='recall',
    n_jobs=-1,
    random_state=42,
    verbose=0
)
search.fit(X_train, y_train)

print("Best recall (RandomCV):", round(search.best_score_, 3))
print("Best parameters:",        search.best_params_)

In [ ]:
best_model = search.best_estimator_
y_prob     = best_model.predict_proba(X_test)[:, 1]

y_pred_default  = best_model.predict(X_test)
y_pred_adjusted = (y_prob >= 0.35).astype(int)

for label, y_pred in [("Threshold = 0.5", y_pred_default), ("Threshold = 0.35", y_pred_adjusted)]:
    print(f"--- RandomCV {label} ---")
    print(f"Accuracy : {accuracy_score(y_test, y_pred):.3f}")
    print(f"ROC-AUC  : {roc_auc_score(y_test, y_prob):.3f}")
    print(classification_report(y_test, y_pred))

### 9b. Feature Importance (after RandomCV tuning)

In [ ]:
model_step    = best_model.named_steps['model']
preprocessor_ = best_model.named_steps['preprocessor']

cat_features_out = (
    preprocessor_.named_transformers_['cat']
    .named_steps['encoder']
    .get_feature_names_out(categorical_cols)
)
feature_names = list(numerical_cols) + list(cat_features_out)

feat_imp = pd.DataFrame({
    'Feature':    feature_names,
    'Importance': model_step.feature_importances_
}).sort_values('Importance', ascending=False)

plt.figure(figsize=(8, 6))
plt.barh(feat_imp['Feature'][:15][::-1], feat_imp['Importance'][:15][::-1])
plt.xlabel("Importance Score")
plt.title("Top 15 Feature Importances — Gradient Boosting")
plt.tight_layout()
plt.show()

### 9c. GridSearchCV (fine-tuned around best RandomCV region)

In [ ]:
cv_grid = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

param_grid = {
    'model__n_estimators':    [150, 200, 250],
    'model__learning_rate':   [0.01, 0.05, 0.1],
    'model__max_depth':       [3, 4, 5],
    'model__subsample':       [0.8, 1.0],
    'model__min_samples_split': [2, 5],
    'model__min_samples_leaf':  [1, 2]
}

grid_search = GridSearchCV(
    pipeline,
    param_grid=param_grid,
    cv=cv_grid,
    scoring='recall',
    n_jobs=-1,
    verbose=0
)
grid_search.fit(X_train, y_train)

print("Best recall (GridSearch):", round(grid_search.best_score_, 3))
print("Best parameters:",          grid_search.best_params_)

### 9d. GridSearchCV — Test Set Evaluation

In [ ]:
grid_best       = grid_search.best_estimator_
y_prob_grid     = grid_best.predict_proba(X_test)[:, 1]

y_pred_grid_default  = grid_best.predict(X_test)
y_pred_grid_adjusted = (y_prob_grid >= 0.35).astype(int)

for label, y_pred in [("Threshold = 0.5", y_pred_grid_default), ("Threshold = 0.35", y_pred_grid_adjusted)]:
    print(f"--- GridSearch {label} ---")
    print(f"Accuracy : {accuracy_score(y_test, y_pred):.3f}")
    print(f"ROC-AUC  : {roc_auc_score(y_test, y_prob_grid):.3f}")
    print(classification_report(y_test, y_pred))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for ax, y_pred, title in zip(
    axes,
    [y_pred_grid_default, y_pred_grid_adjusted],
    ["GB GridSearch (Threshold 0.5)", "GB GridSearch (Threshold 0.35)"]
):
    ConfusionMatrixDisplay(
        confusion_matrix(y_test, y_pred),
        display_labels=["No HTN", "HTN"]
    ).plot(cmap="Blues", ax=ax, colorbar=False)
    ax.set_title(title)
plt.tight_layout()
plt.show()

In [ ]:
fpr, tpr, _ = roc_curve(y_test, y_prob_grid)
auc_score   = roc_auc_score(y_test, y_prob_grid)

plt.figure(figsize=(5, 4))
plt.plot(fpr, tpr, label=f"AUC = {auc_score:.3f}")
plt.plot([0, 1], [0, 1], 'k--')
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve — Gradient Boosting (GridSearch)")
plt.legend()
plt.tight_layout()
plt.show()

## 10. Threshold Analysis

The precision-recall trade-off is evaluated across a range of thresholds to justify the 0.35 selection.  
In a screening context, the cost of a missed hypertensive patient (false negative) outweighs the cost of a false positive.

In [ ]:
thresholds = [0.5, 0.45, 0.40, 0.35, 0.30, 0.25]

threshold_results = []
for t in thresholds:
    y_pred_t = (y_prob >= t).astype(int)
    report   = classification_report(y_test, y_pred_t, output_dict=True)
    threshold_results.append({
        "Threshold": t,
        "Recall":    round(report.get("1.0", report.get(1, {})).get("recall",    0), 3),
        "Precision": round(report.get("1.0", report.get(1, {})).get("precision", 0), 3),
        "F1":        round(report.get("1.0", report.get(1, {})).get("f1-score",  0), 3),
        "Accuracy":  round(accuracy_score(y_test, y_pred_t), 3)
    })

threshold_df = pd.DataFrame(threshold_results)
print("Threshold sweep results:")
display(threshold_df)

In [ ]:
# Visualise recall vs precision across thresholds
plt.figure(figsize=(7, 4))
plt.plot(threshold_df["Threshold"], threshold_df["Recall"],    marker='o', label="Recall")
plt.plot(threshold_df["Threshold"], threshold_df["Precision"], marker='s', label="Precision")
plt.plot(threshold_df["Threshold"], threshold_df["F1"],        marker='^', label="F1")
plt.axvline(0.35, color='red', linestyle='--', alpha=0.7, label="Selected threshold (0.35)")
plt.xlabel("Threshold")
plt.ylabel("Score")
plt.title("Recall / Precision / F1 vs Classification Threshold")
plt.legend()
plt.tight_layout()
plt.show()

## 11. Save Results

In [ ]:
os.makedirs("results", exist_ok=True)

summary = pd.DataFrame([
    {"Stage": "Baseline (0.35)",   "Accuracy": accuracy_score(y_test, y_pred_base),
     "ROC-AUC": roc_auc_score(y_test, y_prob_base),
     "Recall":  recall_score(y_test, y_pred_base)},
    {"Stage": "RandomCV (0.35)",   "Accuracy": accuracy_score(y_test, y_pred_adjusted),
     "ROC-AUC": roc_auc_score(y_test, y_prob),
     "Recall":  recall_score(y_test, y_pred_adjusted)},
    {"Stage": "GridSearch (0.35)", "Accuracy": accuracy_score(y_test, y_pred_grid_adjusted),
     "ROC-AUC": roc_auc_score(y_test, y_prob_grid),
     "Recall":  recall_score(y_test, y_pred_grid_adjusted)},
])

summary.to_excel("results/gb_ripas_summary.xlsx", index=False)
print("Saved: results/gb_ripas_summary.xlsx")
display(summary)